# SkillOS + Gemma 4 on Colab (Free T4 GPU)

This notebook sets up **Ollama + Gemma 4** on a free Colab T4 GPU and exposes it
via a **Cloudflare tunnel** so your local SkillOS instance can use it as a backend.

## Local usage (if you have a GPU locally)

```bash
ollama pull gemma4
python agent_runtime.py --provider gemma interactive
```

## Colab usage

Run all cells below, then copy the tunnel URL and set it locally:

```bash
OLLAMA_BASE_URL=https://something.trycloudflare.com/v1 python agent_runtime.py --provider gemma "Say hello"
```

In [ ]:
# Cell 2: Install Ollama and start the server
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.ai/install.sh | sh

import subprocess, time, os

# Kill any existing Ollama process (needed on re-runs)
!pkill -9 -f "ollama" || true
time.sleep(2)

# Set environment: allow all origins and bind to all interfaces
os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"

# Start Ollama server in background with env vars set via shell
proc = subprocess.Popen(
    "OLLAMA_ORIGINS='*' OLLAMA_HOST='0.0.0.0:11434' ollama serve",
    shell=True,
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(3)
print(f"Ollama server started (PID {proc.pid})")
!curl -s http://localhost:11434/api/version

In [ ]:
# Cell 3: Pull Gemma 4 (e2b quantization fits free T4's 15 GB VRAM)
!ollama pull gemma4:e2b

# Verify model is available
!ollama list

In [ ]:
# Cell 4: Quick test via OpenAI-compatible API
import json, urllib.request

payload = json.dumps({
    "model": "gemma4:e2b",
    "messages": [{"role": "user", "content": "Say hello in one sentence."}],
    "temperature": 0.7,
}).encode()

req = urllib.request.Request(
    "http://localhost:11434/v1/chat/completions",
    data=payload,
    headers={"Content-Type": "application/json"},
)
resp = json.loads(urllib.request.urlopen(req).read())
print(resp["choices"][0]["message"]["content"])

In [ ]:
# Cell 5: Start Cloudflare tunnel to expose Ollama (free, no signup)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import subprocess, re, time

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Read output until we find the tunnel URL
tunnel_url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        break
    print(line.strip())
    match = re.search(r'(https://[\w.-]+\.trycloudflare\.com)', line)
    if match:
        tunnel_url = match.group(1)
        break

if tunnel_url:
    print(f"\n{'='*60}")
    print(f"Tunnel URL: {tunnel_url}")
    print(f"\nUse on your local machine:")
    print(f"  OLLAMA_BASE_URL={tunnel_url}/v1 python agent_runtime.py --provider gemma \"Say hello\"")
    print(f"{'='*60}")
else:
    print("Failed to get tunnel URL. Check output above for errors.")

## Model Variants

| Tag | Params | VRAM | Context | Colab T4? |
|-----|--------|------|---------|-----------|
| `gemma4:e2b` | 12B (Q2) | ~7.2 GB | 128K | Yes |
| `gemma4` | 12B (Q4) | ~9.6 GB | 128K | Yes |
| `gemma4:e4b` | 12B (Q4) | ~9.6 GB | 128K | Yes |
| `gemma4:26b` | 27B (Q4) | ~18 GB | 256K | No (needs A100) |
| `gemma4:31b` | 27B (Q8) | ~20 GB | 256K | No (needs A100) |

## Keeping Alive

- **Colab disconnects** after ~90 min of inactivity. Keep a cell running or use the browser tab.
- **Cloudflare tunnels** are free with no signup. No uptime guarantee but reliable for experimentation.
- Re-run Cell 5 to get a new URL if the tunnel expires.

## SkillOS Usage

```bash
# Set the tunnel URL and run
export OLLAMA_BASE_URL=https://something.trycloudflare.com/v1

# Single goal
python agent_runtime.py --provider gemma "Create a Python hello world script"

# Interactive mode
python agent_runtime.py --provider gemma interactive

# Override model variant
GEMMA_MODEL=gemma4:e2b python agent_runtime.py --provider gemma interactive
```